# 06 · Retratos de Transfermarkt (enlazados, no descargados)

Saca de las 20 páginas de plantilla de Transfermarkt (temporada 2025/26) el nombre, el identificador, la
fecha de nacimiento y la url del retrato de cada jugador, y lo empareja con los 561 del modelo.

**Las imágenes no se descargan ni se republican.** El modelo guarda la url, que Power BI ("URL de imagen") y
Tableau (rol de imagen) cargan directamente desde los servidores de Transfermarkt. Fuente de los retratos:
**Transfermarkt** (transfermarkt.es).

**Peticiones.** Una por página de plantilla, con 5 segundos de pausa y un user-agent identificable. El HTML se
guarda en `data/bruto/transfermarkt/` (no versionado) y no se vuelve a pedir si ya está.

**Emparejamiento**, siempre dentro de los clubes del jugador en la temporada:

- **Claro:** el nombre de Transfermarkt coincide exactamente (sin acentos ni mayúsculas) con el del jugador en
  alguna de las tres fuentes del proyecto, es el único que coincide, y su edad cuadra con la de WhoScored (±1).
- **Confirmado por el autor:** parejas dudosas que el autor ha revisado una a una (lista `CONFIRMADAS_AUTOR`).
- **Dudoso:** el nombre no coincide exacto pero se parece (comparten apellido, uno contiene al otro o son muy
  parecidos) y la edad cuadra; o coincide exacto pero la edad no cuadra. **No se aplica** hasta que el autor
  lo revise.

In [1]:
# Instalamos las librerías del cuaderno; si ya están instaladas no cambia nada
%pip install -q pandas requests unidecode

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importamos os para situarnos en la raíz del proyecto
import os
# Importamos re para leer el HTML
import re
# Importamos time para las pausas
import time
# Importamos html para decodificar entidades
import html
# Importamos difflib para medir el parecido de nombres
import difflib
# Importamos datetime para la fecha de consulta
import datetime
# Importamos requests para las peticiones
import requests
# Importamos pandas para las tablas
import pandas as pd
# Importamos unidecode para comparar nombres sin acentos
from unidecode import unidecode

# Si el cuaderno se ejecuta desde notebooks/, subimos a la raíz para usar rutas relativas
if os.path.basename(os.getcwd()) == "notebooks":
    # Cambiamos a la carpeta superior
    os.chdir("..")

# Ensanchamos la salida para que las tablas no se partan
pd.set_option("display.width", 250)
# Mostramos todas las filas de las tablas de revisión
pd.set_option("display.max_rows", 200)

# User-agent identificable
CABECERAS = {"User-Agent": "scouting-laliga-2526/1.0 (proyecto de datos)"}
# Carpeta del HTML descargado
CARPETA = "data/bruto/transfermarkt"
# La creamos si no existe
os.makedirs(CARPETA, exist_ok=True)

# Identificador de cada club en Transfermarkt, sacado de Wikidata (propiedad P7223)
CLUBES_TM = {
    "Alaves": 1108, "Athletic Club": 621, "Atletico Madrid": 13, "Barcelona": 131, "Celta Vigo": 940,
    "Elche": 1531, "Espanyol": 714, "Getafe": 3709, "Girona": 12321, "Levante": 3368, "Mallorca": 237,
    "Osasuna": 331, "Rayo Vallecano": 367, "Real Betis": 150, "Real Madrid": 418, "Real Oviedo": 2497,
    "Real Sociedad": 681, "Sevilla": 368, "Valencia": 1049, "Villarreal": 1050,
}

## 1. Las 20 plantillas

In [3]:
# Recorremos los clubes
for equipo, id_club in CLUBES_TM.items():
    # Fichero local de la página
    ruta = os.path.join(CARPETA, f"{id_club}.html")
    # Si ya está, no se vuelve a pedir
    if os.path.exists(ruta):
        continue
    # Página de plantilla detallada de la temporada 2025/26
    url = f"https://www.transfermarkt.es/club/kader/verein/{id_club}/saison_id/2025/plus/1"
    # Petición
    r = requests.get(url, headers=CABECERAS, timeout=60)
    # Si no responde bien, paramos: no se insiste ni se esquiva ningún bloqueo
    assert r.status_code == 200, f"{equipo}: respuesta {r.status_code}"
    # Guardamos el HTML
    open(ruta, "w", encoding="utf-8").write(r.text)
    # Informamos
    print(f"{equipo:<16} {r.status_code} {len(r.text) // 1024} KB")
    # Pausa entre peticiones
    time.sleep(5)

# Jugadores de todas las plantillas
filas = []
for equipo, id_club in CLUBES_TM.items():
    # HTML de la plantilla
    h = open(os.path.join(CARPETA, f"{id_club}.html"), encoding="utf-8").read()
    # Tabla de jugadores
    tabla = h[h.find('class="items"'):]
    # Una fila por jugador
    for fila in re.findall(r'<tr class="(?:odd|even)">(.*?)(?=<tr class="(?:odd|even)">|</tbody>)', tabla, re.S):
        # Identificador y nombre del enlace al perfil (el enlace puede llevar iconos de capitán o de lesión dentro)
        perfil = re.search(r'href="/[^"]+/profil/spieler/(\d+)">(.*?)</a>', fila, re.S)
        # Url del retrato
        retrato = re.search(r'data-src="(https://img\.a\.transfermarkt\.technology/portrait/[^"]+)"', fila)
        # Fecha de nacimiento y edad
        nac = re.search(r'>(\d{2}/\d{2}/\d{4}) \((\d+)\)<', fila)
        # Posición
        pos = re.search(r'<td>\s*([^<]+?)\s*</td>\s*</tr>\s*</table>', fila)
        # Sin perfil no es una fila de jugador
        if not perfil:
            continue
        # Guardamos el jugador
        filas.append({"equipo": equipo, "id_transfermarkt": int(perfil.group(1)),
                      "nombre_tm": html.unescape(re.sub(r"<[^>]+>", "", perfil.group(2))).replace(" ", " ").strip(), "url_retrato": retrato.group(1) if retrato else None,
                      "nacimiento_tm": nac.group(1) if nac else None, "edad_tm": int(nac.group(2)) if nac else None,
                      "posicion_tm": pos.group(1) if pos else None})
# Pasamos a tabla
tm = pd.DataFrame(filas)
# Retratos genéricos de Transfermarkt (silueta sin foto)
tm["retrato_generico"] = tm["url_retrato"].fillna("").str.contains("default")
# Resumen
print("jugadores en las 20 plantillas:", len(tm), "· con retrato propio:", int((~tm["retrato_generico"] & tm["url_retrato"].notna()).sum()))
print("porteros:", int(tm["posicion_tm"].eq("Portero").sum()))

jugadores en las 20 plantillas: 791 · con retrato propio: 705
porteros: 89


## 2. Emparejamiento con los 561 del modelo

In [4]:
# Normaliza un nombre como en el cuaderno 00
def normalizar(nombre):
    # Sin acentos, en minúsculas, sin puntos ni guiones
    n = unidecode(str(nombre)).lower().replace(".", "").replace("-", " ")
    # Espacios compactados
    return " ".join(n.split())


# Nombre normalizado en Transfermarkt
tm["n"] = tm["nombre_tm"].map(normalizar)

# Jugadores del modelo con su equipo
jug = pd.read_csv("data/modelo/dim_jugador.csv", dtype={"id_whoscored": "Int64"}).merge(
    pd.read_csv("data/modelo/dim_equipo.csv")[["id_equipo", "equipo"]], on="id_equipo")
# Nombres en las otras fuentes
jug = jug.merge(pd.read_csv("data/mapa_nombres.csv")[["id_understat", "nombre_understat", "nombre_whoscored", "equipo_understat"]],
                on="id_understat")
# Edad según WhoScored
edad = pd.read_csv("data/origen/alineaciones_laliga_2526.csv").groupby("player_id")["edad"].max()
# La añadimos
jug["edad_ws"] = jug["id_whoscored"].map(edad)


# Clubes del jugador en la temporada
def clubes(f):
    # El del modelo y el de Understat
    c = {f["equipo"], f["equipo_understat"]}
    # Más los del reparto de eventos ("Celta Vigo 10 + Girona 17")
    if pd.notna(f["equipos_temporada"]):
        c |= {re.sub(r"\s+\d+$", "", p) for p in f["equipos_temporada"].split(" + ")}
    # Conjunto de clubes
    return c


# ¿Se parecen dos nombres normalizados?
def parecidos(a, b):
    # Palabras de cada uno
    ta, tb = a.split(), b.split()
    # Comparten la última palabra (apellido), uno contiene al otro, o son muy parecidos
    return ta[-1] == tb[-1] or a in b or b in a or difflib.SequenceMatcher(None, a, b).ratio() >= 0.8


# ¿Cuadra la edad? (vacío si no hay dato para comparar)
def edad_cuadra(f, c):
    # Sin alguna de las dos edades no se puede comprobar
    if pd.isna(f["edad_ws"]) or pd.isna(c["edad_tm"]):
        return None
    # Margen de un año
    return abs(int(f["edad_ws"]) - int(c["edad_tm"])) <= 1


# Empareja un jugador del modelo; devuelve estado, candidato y motivo
def emparejar(f):
    # Nombres del jugador en las tres fuentes
    nombres = {normalizar(x) for x in [f["jugador"], f["nombre_understat"], f["nombre_whoscored"]] if pd.notna(x)}
    # Jugadores de Transfermarkt de sus clubes
    pool = tm[tm["equipo"].isin(clubes(f))].drop_duplicates("id_transfermarkt")
    # Coincidencias exactas
    exactos = pool[pool["n"].isin(nombres)]
    # Un único exacto
    if len(exactos) == 1:
        # Candidato
        c = exactos.iloc[0]
        # Comprobación de edad
        ok = edad_cuadra(f, c)
        # Claro si la edad cuadra o no se puede comprobar
        if ok is not False:
            return "claro", c, "nombre exacto" + ("" if ok else ", edad sin comprobar")
        # Exacto pero con edad distinta: dudoso
        return "dudoso", c, f"nombre exacto, pero edad {int(c['edad_tm'])} en Transfermarkt y {int(f['edad_ws'])} en WhoScored"
    # Varios exactos: dudoso
    if len(exactos) > 1:
        return "dudoso", exactos.iloc[0], f"{len(exactos)} jugadores con el mismo nombre"
    # Nombres parecidos con edad que cuadra
    similares = pool[[any(parecidos(n, x) for n in nombres) for x in pool["n"]]]
    # Nos quedamos con los de edad compatible
    similares = similares[[edad_cuadra(f, c) is not False for _, c in similares.iterrows()]]
    # Uno solo: dudoso, para revisar
    if len(similares) == 1:
        c = similares.iloc[0]
        return "dudoso", c, f"nombre parecido, edad {('%d/%d' % (c['edad_tm'], f['edad_ws'])) if edad_cuadra(f, c) else 'sin comprobar'}"
    # Varios parecidos: dudoso con el primero, avisando
    if len(similares) > 1:
        return "dudoso", similares.iloc[0], f"{len(similares)} candidatos parecidos: " + ", ".join(similares["nombre_tm"])
    # Nada
    return "sin_coincidencia", None, "no está en las plantillas de sus clubes"


# Emparejamos a los 561
res = [emparejar(f) for _, f in jug.iterrows()]
# Estado
jug["estado"] = [r[0] for r in res]
# Motivo
jug["motivo"] = [r[2] for r in res]
# Datos del candidato
for col in ["id_transfermarkt", "nombre_tm", "url_retrato", "edad_tm", "retrato_generico"]:
    # Valor o vacío
    jug[col] = [r[1][col] if r[1] is not None else None for r in res]
# Parejas dudosas revisadas y confirmadas por el autor el 16/11/2026: (jugador del modelo, equipo, nombre en Transfermarkt)
CONFIRMADAS_AUTOR = [
    ("Copete", "Valencia", "José Copete"),
    ("Javier Morcillo", "Atletico Madrid", "Javi Morcillo"),
    ("Valde", "Real Madrid", "Víctor Valdepeñas"),
    ("Cubo", "Atletico Madrid", "Miguel Cubo"),
    ("Lancinet Kourouma", "Girona", "Lass Kourouma"),
    ("Adrià Alti", "Villarreal", "Adrià Altimira"),
    ("Tomás Marqués", "Barcelona", "Tommy Marqués"),
    ("Manuel Serrano", "Real Madrid", "Manu Serrano"),
]
# Recorremos las confirmadas
for jugador, equipo, nombre_tm in CONFIRMADAS_AUTOR:
    # Fila del jugador, que tiene que estar dudosa y con ese mismo candidato
    sel = (jug["jugador"] == jugador) & (jug["equipo"] == equipo) & (jug["estado"] == "dudoso") & (jug["nombre_tm"] == nombre_tm)
    # Si no se encuentra exactamente una, paramos
    assert sel.sum() == 1, (jugador, equipo)
    # Pasa a confirmada por el autor
    jug.loc[sel, "estado"] = "confirmado_autor"
# Sergio Lozano (Rayo) se queda sin aplicar: es el cruce dudoso que ya está en data/cruces_descartados.csv

# Un identificador de Transfermarkt no puede ir a dos jugadores claros o confirmados
claros = jug[jug["estado"].isin(["claro", "confirmado_autor"])]
assert claros["id_transfermarkt"].is_unique, "un jugador de Transfermarkt emparejado dos veces"
# Recuento
print(jug["estado"].value_counts().to_string())

estado
claro               547
confirmado_autor      8
sin_coincidencia      5
dudoso                1


In [5]:
# Parejas dudosas, para revisión del autor
print(jug[jug["estado"] == "dudoso"][["jugador", "nombre_understat", "nombre_whoscored", "nombre_tm", "equipo",
                                      "minutos", "edad_ws", "edad_tm", "motivo"]].sort_values("equipo").to_string(index=False))

      jugador nombre_understat nombre_whoscored     nombre_tm         equipo  minutos  edad_ws  edad_tm                          motivo
Sergio Lozano    Sergio Lozano              NaN Sergio Lozano Rayo Vallecano        7      NaN     27.0 2 jugadores con el mismo nombre


In [6]:
# Jugadores sin coincidencia, por minutos
print(jug[jug["estado"] == "sin_coincidencia"][["jugador", "equipo", "minutos", "edad_ws"]].sort_values(
    "minutos", ascending=False).to_string(index=False))

            jugador  equipo  minutos  edad_ws
Adrián de la Fuente Levante     2704     27.0
      Djené Dakonam  Getafe     2601     34.0
   Carlos Benavídez  Alaves      311      NaN
        Yago Alonso   Elche      275      NaN
   Adam El Mokhtari   Elche       85      NaN


## 3. Guardado

Se aplican los claros y los confirmados por el autor que tienen retrato propio. Los que solo tienen la silueta
genérica de Transfermarkt no se aplican: en el panel llevarán el retrato de iniciales, igual que los dudosos
y los que no tienen coincidencia.

In [7]:
# Se aplica si es claro y tiene retrato que no es la silueta genérica
jug["aplicado"] = jug["estado"].isin(["claro", "confirmado_autor"]) & jug["url_retrato"].notna() & ~jug["retrato_generico"].fillna(False).astype(bool)
# Fecha de consulta
jug["fecha_consulta"] = datetime.date.today().isoformat()
# Columnas del fichero
COLS = ["id_jugador", "jugador", "equipo", "minutos", "id_transfermarkt", "nombre_tm", "edad_ws", "edad_tm",
        "url_retrato", "retrato_generico", "estado", "motivo", "aplicado", "fecha_consulta"]
# Guardamos
jug.sort_values("id_jugador")[COLS].to_csv("data/fotos_transfermarkt.csv", index=False)
# Resumen
print("aplicados:", int(jug["aplicado"].sum()), "· claros o confirmados con silueta genérica:",
      int((jug["estado"].isin(["claro", "confirmado_autor"]) & jug["retrato_generico"].fillna(False).astype(bool)).sum()))

aplicados: 537 · claros o confirmados con silueta genérica: 18


## 4. ¿Funciona el enlazado y cumple los límites de Tableau?

Tableau exige que la imagen pese menos de 128 KB y que la url tenga menos de 255 caracteres. Para cada retrato
aplicado se hace una petición HEAD (solo cabeceras, sin descargar la imagen) **sin referente**, como la haría
un panel alojado fuera de Transfermarkt.

In [8]:
# Retratos aplicados
aplicados = jug[jug["aplicado"]].copy()
# Resultado de cada comprobación
estado_http, peso = [], []
# Recorremos las url
for u in aplicados["url_retrato"]:
    # Petición HEAD sin referente
    r = requests.head(u, headers=CABECERAS, timeout=30, allow_redirects=True)
    # Código de respuesta
    estado_http.append(r.status_code)
    # Peso declarado en KB
    peso.append(round(int(r.headers.get("content-length", 0)) / 1024, 1))
    # Pausa corta
    time.sleep(0.2)
# Guardamos los resultados
aplicados["http"], aplicados["kb"] = estado_http, peso
# Resumen
print("respuestas:", aplicados["http"].value_counts().to_dict())
print(f"peso: mínimo {aplicados['kb'].min()} KB · máximo {aplicados['kb'].max()} KB · mediana {aplicados['kb'].median()} KB")
print("por encima de 128 KB:", int((aplicados["kb"] >= 128).sum()))
print("url de 255 caracteres o más:", int((aplicados["url_retrato"].str.len() >= 255).sum()),
      "· la más larga:", int(aplicados["url_retrato"].str.len().max()))
# Añadimos peso y código al fichero
jug = jug.merge(aplicados[["id_jugador", "http", "kb"]], on="id_jugador", how="left")
# Volvemos a guardar con esas dos columnas
jug.sort_values("id_jugador")[COLS + ["http", "kb"]].to_csv("data/fotos_transfermarkt.csv", index=False)

respuestas: {200: 537}
peso: mínimo 2.9 KB · máximo 30.3 KB · mediana 3.8 KB
por encima de 128 KB: 0
url de 255 caracteres o más: 0 · la más larga: 85
